# Using a cross encoder to optimize scoring

As you have seen in the [previous notebook](22-similarity-rrf.ipynb), reciprocal
rank fusion can be used to combine multiple result lists into one. As the
scores are incommensurable, only the rank was considered for this.

However, we can improve on that by using a [cross encoder](https://sbert.net/examples/cross_encoder/applications/README.html).
The cross encoder is a (modified) BERT model which checks the similarity of two
sentences or how well an answer fits to a question. By using the last layer of
the BERT transformer network, a probability can be extracted and will be used as
a score.

There are many cross encoders (sometimes also called rerankers) available. You can find a list on 
[Hugging Face](https://huggingface.co/models?pipeline_tag=text-ranking). Be careful,
not all have the same calling conventions and the best models are often quite slow
and mostly suitable for offline use.

## Load data (from previous notebook)

In [1]:
import json
with open("sentences.json") as f:
    sentences = json.load(f)

In [2]:
len(sentences)

18342

In [3]:
import numpy as np
with open("sentences-mqa.npy", "rb") as f:
    sembeddings = np.load(f)

## Retrieval

In [4]:
import numpy as np
import pandas as pd
def search(query, text, corpus_embeddings, bi_encoder, cross_encoder, query_prompt_name=None, top_k=100):
    # code query to restrict search space
    question_embedding = bi_encoder.encode(query, normalize_embeddings=True, prompt_name=query_prompt_name)
    
    # Determine similarity (vectors are normalized)
    sim = bi_encoder.similarity(question_embedding, corpus_embeddings)[0].numpy() 
    
    # Get most similar top_k by sorting
    hits = [ { "id": i, "text": text[i], "score": sim[i] } 
                     for i in sim.argsort()[::-1][0:top_k] ]

    # Consider only top hits for re-ranking
    cross_input = [[query, hit["text"]] for hit in hits]
    # cross-encode (this takes most time)
    cross_scores = cross_encoder.predict(cross_input)

    # Integrate cross-scores in original hits (this would be easier with pandas)
    for i in range(len(cross_scores)):
        hits[i]["cross-score"] = cross_scores[i]

    # nre-sort by cross-score, descending!
    hits = sorted(hits, key=lambda x: x["cross-score"], reverse=True)
    
    # Return top-20 results of re-ranker as dataframe
    return pd.DataFrame(hits[0:20]).set_index("id")

In [5]:
# bi-encoder is needed
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# cross encoder
from sentence_transformers import CrossEncoder, util
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [7]:
pd.set_option('display.max_colwidth', 0)

In [8]:
search("Is the climate crisis worse in poorer countries?", 
       sentences, sembeddings, model, cross_encoder).style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
8653,"Climate change is causing geopolitical shifts in agriculture and fisheries, worsening the crises in countries that are vulnerable to food shortages.",0.614005,2.545639
6246,Pakistan is one of the countries worst affected by the impacts of climate change.,0.602407,1.197320
864,"Our problems as poor countries tend to also become the problems of rich countries — demographic disparity, economic disparity, climate deregulation and the loss of institutional trust, among other things.",0.605697,1.026165
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.617619,0.761499
8734,"As a result, countries like mine, which are unable to access concessionary funding, are forced to fix the climate crisis by obtaining loans at exorbitant rates from the very countries where the problem originated.",0.584111,0.644702
16736,"We know that many States here, in particular the most vulnerable — those that have contributed the least to global warming and have burned the least fossil fuels — are the ones that have suffered the most from the climate crisis.",0.618090,0.313781
1574,"The climate problem is worsening as the world’s natural carbon sinks, such as our oceans and rainforests, cease to spawn life.",0.630825,0.014639
1778,"As with many conflicts, the main cause of climate crisis is a lack of trust and solidarity, coupled with the selfishness of some countries.",0.637547,-0.035266
13766,"The effects of climate change are causing suffering to the most vulnerable communities, especially small island developing States, least developed countries and those affected by conflict.",0.609744,-0.114798


In [9]:
# try with different model
# if you believe the cross encoder, the document with the lowest cross encoding
# score might be a measure of the quality of the embedding model
model2 = SentenceTransformer('google/embeddinggemma-300m')
with open("sentences-gemma.npy", "rb") as f:
    sembeddings2 = np.load(f)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [10]:
search("Is the climate crisis worse in poorer countries?", 
       sentences, sembeddings2, model2, cross_encoder).style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
8653,"Climate change is causing geopolitical shifts in agriculture and fisheries, worsening the crises in countries that are vulnerable to food shortages.",0.564359,2.545638
6246,Pakistan is one of the countries worst affected by the impacts of climate change.,0.551488,1.197322
864,"Our problems as poor countries tend to also become the problems of rich countries — demographic disparity, economic disparity, climate deregulation and the loss of institutional trust, among other things.",0.582749,1.026165
14458,"As the global climate change crisis continues to worsen, so too will the issue of food insecurity and malnutrition.",0.603036,0.843401
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.580157,0.761496
16736,"We know that many States here, in particular the most vulnerable — those that have contributed the least to global warming and have burned the least fossil fuels — are the ones that have suffered the most from the climate crisis.",0.546305,0.313781
13766,"The effects of climate change are causing suffering to the most vulnerable communities, especially small island developing States, least developed countries and those affected by conflict.",0.612487,-0.114798
8650,The climate crisis is another challenge that exacerbates the economic divide between nations and impedes humankind’s sustainable development.,0.552691,-0.342627
2771,The climate emergency is worsening.,0.629654,-0.415865


In [12]:
model3 = SentenceTransformer("microsoft/harrier-oss-v1-0.6b", trust_remote_code=True)
with open("sentences-harrier.npy", "rb") as f:
    sembeddings3 = np.load(f)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [13]:
search("Is the climate crisis worse in poorer countries?", 
       sentences, sembeddings3, model3, cross_encoder, query_prompt_name="web_search_query").style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
8653,"Climate change is causing geopolitical shifts in agriculture and fisheries, worsening the crises in countries that are vulnerable to food shortages.",0.565997,2.545639
7400,"SIDs, though not the poorest countries, are extremely vulnerable to climate and other shocks, and lack resilience due to structural problems of limited human and financial resources, lack of economies of scale and higher costs due to their isolation from major manufacturing hubs.",0.629053,2.120155
8853,"In that composite package of policies and measures, special consideration has to be accorded to the most vulnerable countries, such as small island developing States in the Caribbean and the Pacific and the poorer communities in climate-distressed areas of Africa.",0.583743,2.009091
6246,Pakistan is one of the countries worst affected by the impacts of climate change.,0.582765,1.197322
864,"Our problems as poor countries tend to also become the problems of rich countries — demographic disparity, economic disparity, climate deregulation and the loss of institutional trust, among other things.",0.623622,1.026165
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.619098,0.761499
8734,"As a result, countries like mine, which are unable to access concessionary funding, are forced to fix the climate crisis by obtaining loans at exorbitant rates from the very countries where the problem originated.",0.628165,0.644702
2496,"The cross-border financial impacts of crises, such as climate change and the pandemic, are impeding the ability of smaller indebted countries, such as mine, to make progress on the SDGs and climate adaptation and mitigation.",0.606046,0.466476
4635,"Loss and damage caused by the climate crisis are accumulating every day in Micronesia, and they will continue to worsen at a faster pace as tipping points are reached.",0.588539,0.418001


## Alternative cross encoder

In [14]:
cross_encoder2 = CrossEncoder('mixedbread-ai/mxbai-rerank-large-v1')

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [15]:
search("Is the climate crisis worse in poorer countries?", sentences, sembeddings, model,
       cross_encoder2).style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
2776,"Despite having contributed the least to climate change, it is the poorest and most vulnerable parts of the world that suffer the most devastating consequences.",0.695466,0.973145
10737,"Developing countries, such as Cote d’Ivoire, which are only marginally responsible for climate change, are disproportionately affected and are suffering the most from its consequences.",0.661557,0.959961
8346,"Developing countries, particularly least developed countries, are currently the most vulnerable to the severe consequences of climate change, natural disasters and diseases.",0.643064,0.950195
16512,"We also acknowledge that our people, the people of the small island developing States, those who are least culpable for the climate crisis, are the ones who continue to be most disproportionately affected.",0.596676,0.950195
4419,"It is also no secret that those who are least responsible for climate change are the ones suffering the most from its effects, particularly small island developing States.",0.617711,0.944336
4543,It is the vulnerable populations in the global South who are most affected by the loss and damage caused by climate change.,0.591176,0.939453
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.617619,0.938477
13766,"The effects of climate change are causing suffering to the most vulnerable communities, especially small island developing States, least developed countries and those affected by conflict.",0.609744,0.936523
16736,"We know that many States here, in particular the most vulnerable — those that have contributed the least to global warming and have burned the least fossil fuels — are the ones that have suffered the most from the climate crisis.",0.618090,0.921875


In [16]:
search("Is the climate crisis worse in poorer countries?", sentences, sembeddings2, model2, 
       cross_encoder2).style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
2776,"Despite having contributed the least to climate change, it is the poorest and most vulnerable parts of the world that suffer the most devastating consequences.",0.656557,0.973145
10737,"Developing countries, such as Cote d’Ivoire, which are only marginally responsible for climate change, are disproportionately affected and are suffering the most from its consequences.",0.571125,0.959961
8346,"Developing countries, particularly least developed countries, are currently the most vulnerable to the severe consequences of climate change, natural disasters and diseases.",0.589566,0.950195
16512,"We also acknowledge that our people, the people of the small island developing States, those who are least culpable for the climate crisis, are the ones who continue to be most disproportionately affected.",0.530645,0.950195
11642,"The 3.3 billion people in those counties are trapped in a vicious cycle of emergency responses, reconstruction and recovery from more frequent climate shocks, which diverts resources away from both development and climate action and sucks vulnerable countries into a downward spiral of debt and environmental stress.",0.536774,0.946289
4419,"It is also no secret that those who are least responsible for climate change are the ones suffering the most from its effects, particularly small island developing States.",0.595041,0.944336
10201,We remain concerned that countries that have contributed less to the global emission of greenhouse gases continue to be disproportionately affected by the impacts of climate change.,0.628874,0.943359
4543,It is the vulnerable populations in the global South who are most affected by the loss and damage caused by climate change.,0.571483,0.939453
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.580157,0.938477


In [17]:
search("Is the climate crisis worse in poorer countries?", sentences, sembeddings3, model3, 
       cross_encoder2, query_prompt_name="query").style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
2776,"Despite having contributed the least to climate change, it is the poorest and most vulnerable parts of the world that suffer the most devastating consequences.",0.668364,0.973145
10737,"Developing countries, such as Cote d’Ivoire, which are only marginally responsible for climate change, are disproportionately affected and are suffering the most from its consequences.",0.634316,0.959961
8346,"Developing countries, particularly least developed countries, are currently the most vulnerable to the severe consequences of climate change, natural disasters and diseases.",0.621781,0.950195
476,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.672699,0.938477
17117,"When we put the clean energy transition at the heart of the fight against climate change on the global level, we should not forget that the most vulnerable communities, which have historically contributed the least to climate change, are often the ones most and worst affected — both by climate conditions and by the costs of the green energy transition as a remedy.",0.633871,0.926758
16736,"We know that many States here, in particular the most vulnerable — those that have contributed the least to global warming and have burned the least fossil fuels — are the ones that have suffered the most from the climate crisis.",0.661278,0.921875
4039,"We are facing an existential climate catastrophe for millions of people, especially our brothers and sisters from small island developing States, in both the Caribbean and the Pacific.",0.627430,0.920410
10745,"Such devastation — the cost of our inaction on climate change — is erasing the progress made towards achieving the SDGs, especially for the countries that are most vulnerable to the effects of global warming.",0.625334,0.913574
8732,The behemoth industrial countries and companies are pushing small island developing States (SIDS) and others onto the front lines of climate change.,0.628083,0.909180


In [18]:
search("Which country is affected most by climate change?", sentences, sembeddings3, model3, 
       cross_encoder2, query_prompt_name="web_search_query").style.background_gradient(cmap='coolwarm')

,text,score,cross-score
id,,,
2136,Climate vulnerable mountainous countries like Nepal have been bearing the severe brunt of climate change.,0.591030,0.968262
6246,Pakistan is one of the countries worst affected by the impacts of climate change.,0.660319,0.962891
4106,Tajikistan is experiencing the impact of climate change first-hand and is considered one of the countries most vulnerable to the impact of climate change.,0.644951,0.960449
5819,"Andorra is a high mountain country and particularly sensitive to climate change, which seriously threatens our biodiversity, our water cycle and even, ultimately, our way of life.",0.553615,0.934082
10506,"Tuvalu is particularly vulnerable to the adverse impacts of climate change, such as extreme weather events, which affect our ability to achieve our SDG targets.",0.573730,0.932129
10737,"Developing countries, such as Cote d’Ivoire, which are only marginally responsible for climate change, are disproportionately affected and are suffering the most from its consequences.",0.616913,0.931152
16755,Central America and the Caribbean that are suffering the most devastating effects of climate change.,0.631017,0.924805
17309,"We in the Pacific directly bear the brunt of climate change on our coastlines, our communities, our livelihoods, our security and, indeed, our very statehood and identity.",0.554798,0.920898
8571,Bangladesh is one of the most climatically vulnerable countries in the world.,0.613350,0.916992
